# Projeto — Atividade de Internamento Hospitalar em Portugal

## Etapas do projeto:
1. Preparar os dados (ler CSV → limpar → explorar)
2. Guardar numa base de dados SQL e fazer consultas
3. Analisar e visualizar os resultados

# ESTAÇÃO 1 — Preparar os dados (ler CSV e explorar)

In [3]:


import sqlite3
from pathlib import Path
import pandas as pd

df = pd.read_csv("../data/raw/atividade-de-internamento-hospitalar_fich1.csv",
                  sep=";", encoding="utf-8-sig")

# Renomear colunas para snake_case sem acentos/espaços
df.columns = ["periodo", "regiao","instituicao","localizacao","especialidade","doentes_saidos","dias_internamento"]



print(df.shape)          # (linhas, colunas)
print(df.columns)        # nomes das colunas
display(df.head())         # primeiras 5 linhas
display(df.describe().round(2))     # estatísticas básicas das colunas numéricas
print(df.isna().sum())   # quantos valores em falta por coluna
display(df.info()) 
print(f"Linhas em bruto: {len(df):,}")# len(df) = nº de linhas.  O  f"...{x:,}"  mostra o número com separador de milhares.



(17616, 7)
Index(['periodo', 'regiao', 'instituicao', 'localizacao', 'especialidade',
       'doentes_saidos', 'dias_internamento'],
      dtype='str')


,periodo,regiao,instituicao,localizacao,especialidade,doentes_saidos,dias_internamento
0,2015-05,Região de Saúde LVT,"Centro Hospitalar de Lisboa Ocidental, EPE","38.708454, -9.216985",Outras Camas,329.0,8327.0
1,2015-05,Região de Saúde Norte,"Centro Hospitalar Trás-os-Montes e Alto Douro,...","41.3031784, -7.7515252",Outras Camas,150.0,2392.0
2,2015-05,Região de Saúde Norte,"Hospital da Senhora da Oliveira, Guimarães, EPE","41.4387173, -8.3086907",Outras Camas,116.0,2066.0
3,2015-05,Região de Saúde Norte,"Hospital de Braga, PPP","41.56785, -8.398982",Outras Camas,71.0,1470.0
4,2015-05,Região de Saúde Norte,"Unidade Local de Saúde de Matosinhos, EPE","41.1794456, -8.6745115",Outras Camas,49.0,802.0


,doentes_saidos,dias_internamento
count,17588.00,17583.00
mean,3157.16,27088.15
std,3942.83,32915.38
min,0.00,37.00
25%,420.75,4929.00
50%,1710.00,15470.00
75%,4473.25,36047.50
max,34354.00,259313.00


periodo               0
regiao                0
instituicao           0
localizacao           0
especialidade         0
doentes_saidos       28
dias_internamento    33
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 17616 entries, 0 to 17615
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   periodo            17616 non-null  str    
 1   regiao             17616 non-null  str    
 2   instituicao        17616 non-null  str    
 3   localizacao        17616 non-null  str    
 4   especialidade      17616 non-null  str    
 5   doentes_saidos     17588 non-null  float64
 6   dias_internamento  17583 non-null  float64
dtypes: float64(2), str(5)
memory usage: 963.5 KB


None

Linhas em bruto: 17,616


# ESTAÇÃO 2 — Guardar numa base de dados e consultar com SQL

O objetivo é guardar os dados de forma estruturada numa base de dados e realizar consultas SQL para explorar a informação.

## Análise da demória média de internamento por região

 Como gestor de uma unidade hospitalar, quero ver a demora média de internamento da minha região comparada com as restantes regiões de saúde, para identificar se o meu hospital está acima ou abaixo da média nacional.

In [ ]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
con = sqlite3.connect("../data/processed/internamento-hospital.db")


# df.to_sql escreve o DataFrame como uma TABELA dentro da base de dados.
# if_exists='replace' substitui a tabela se já existir; index=False não guarda o nº da linha.

df.to_sql("internamento", con, if_exists="replace", index=False)

q = """

SELECT regiao,        
ROUND(SUM(dias_internamento) / SUM(doentes_saidos)) AS demora_media_regiao
FROM internamento
WHERE doentes_saidos > 0
GROUP BY regiao
ORDER BY demora_media_regiao DESC;

"""
print(" ")
print("1ª QUERY | Analisar a demora média por região")
display(pd.read_sql(q, con))



 
1ª QUERY | Analisar a demora média por região


,regiao,demora_media_regiao
0,Região de Saúde do Algarve,10.0
1,Região de Saúde LVT,9.0
2,Região de Saúde do Centro,8.0
3,Região de Saúde do Alentejo,8.0
4,Região de Saúde Norte,8.0


## Análise da evolução dos dias de internamento ao longo do tempo 

Como responsável de planeamento da ACSS, quero ver a evolução dos dias de internamento ao longo do tempo (2015–2026), para perceber o impacto de eventos como a pandemia na capacidade hospitalar.

In [14]:
q = """

SELECT SUBSTR(periodo, 1, 4) AS ano,     
ROUND(SUM(dias_internamento) / SUM(doentes_saidos)) AS demora_media
FROM internamento
WHERE doentes_saidos > 0
GROUP BY ano
ORDER BY ano;
"""

print("2ª QUERY | Analisar a evolução dos dias de internamento")
display(pd.read_sql(q, con))

2ª QUERY | Analisar a evolução dos dias de internamento


,ano,demora_media
0,2015,8.0
1,2016,8.0
2,2017,8.0
3,2018,9.0
4,2019,9.0
5,2020,9.0
6,2021,9.0
7,2022,9.0
8,2023,9.0
9,2024,9.0
